### 0. Imports and Paths

In [1]:
from pathlib import Path
import numpy as np
from scipy.stats import wilcoxon

In [2]:
DATA_ROOT = Path("../error_analysis")

GROUND_TRUTH_DIR = DATA_ROOT / "ground_truth_labels"
YOLOV5M_DIR = DATA_ROOT / "pred_yolov5m_labels"
YOLOV8M_DIR = DATA_ROOT / "pred_yolov8m_labels"
YOLO26M_DIR = DATA_ROOT / "pred_yolo26m_labels"

IOU_THRESHOLD = 0.5

### 1. Helper functions

In [ ]:
def load_yolo_labels(path, has_conf=False):
    boxes = []
    if not path.exists():
        return boxes

    with open(path, "r") as f:
        for line in f:
            parts = list(map(float, line.strip().split()))
            if has_conf:
                cls, x, y, w, h, conf = parts
            else:
                cls, x, y, w, h = parts
            boxes.append((int(cls), x, y, w, h))
    return boxes

In [4]:
def yolo_to_xyxy(x, y, w, h):
    x1 = x - w / 2
    y1 = y - h / 2
    x2 = x + w / 2
    y2 = y + h / 2
    return x1, y1, x2, y2

In [5]:
def compute_iou(box1, box2):
    x1_1, y1_1, x2_1, y2_1 = yolo_to_xyxy(*box1[1:])
    x1_2, y1_2, x2_2, y2_2 = yolo_to_xyxy(*box2[1:])

    xi1 = max(x1_1, x1_2)
    yi1 = max(y1_1, y1_2)
    xi2 = min(x2_1, x2_2)
    yi2 = min(y2_1, y2_2)

    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    area1 = (x2_1 - x1_1) * (y2_1 - y1_1)
    area2 = (x2_2 - x1_2) * (y2_2 - y1_2)

    union = area1 + area2 - inter_area
    return inter_area / union if union > 0 else 0

In [6]:
def compute_image_metrics(gt_boxes, pred_boxes):
    matched_gt = set()
    matched_pred = set()

    for i, gt in enumerate(gt_boxes):
        for j, pred in enumerate(pred_boxes):
            if j in matched_pred:
                continue
            if gt[0] != pred[0]:
                continue
            if compute_iou(gt, pred) >= IOU_THRESHOLD:
                matched_gt.add(i)
                matched_pred.add(j)
                break

    TP = len(matched_gt)
    FP = len(pred_boxes) - TP
    FN = len(gt_boxes) - TP

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    return precision, recall, f1

In [10]:
def collect_metrics(pred_dir, conf):
    precisions, recalls, f1s = [], [], []

    for gt_file in sorted(GROUND_TRUTH_DIR.glob("*.txt")):
        pred_file = pred_dir / gt_file.name

        gt_boxes = load_yolo_labels(gt_file)
        pred_boxes = load_yolo_labels(pred_file, has_conf=conf)

        p, r, f1 = compute_image_metrics(gt_boxes, pred_boxes)

        precisions.append(p)
        recalls.append(r)
        f1s.append(f1)

    return np.array(precisions), np.array(recalls), np.array(f1s)

### 2. Executing Wilcoxon Test

In [11]:
p5, r5, f15 = collect_metrics(YOLOV5M_DIR, conf=True)
p8, r8, f18 = collect_metrics(YOLOV8M_DIR, conf=False)
p26, r26, f126 = collect_metrics(YOLO26M_DIR, conf=False)

In [ ]:
def wilcoxon_test(a, b, name_a, name_b):
    stat, p = wilcoxon(a, b)
    n = len(a)
    z = (stat - n*(n+1)/4) / np.sqrt(n*(n+1)*(2*n+1)/24)
    r = abs(z) / np.sqrt(n)

    print(f"{name_a} vs {name_b}")
    print(f"  p-value: {p:.4f}")
    print(f"  effect size r: {r:.3f}\n")

In [20]:
wilcoxon_test(f18, f15, "YOLOv8m", "YOLOv5m")
wilcoxon_test(f18, f126, "YOLOv8m", "YOLO26m")
wilcoxon_test(f126, f15, "YOLO26m", "YOLOv5m")

YOLOv8m vs YOLOv5m
  p-value: 0.2792
  effect size r: 0.500

YOLOv8m vs YOLO26m
  p-value: 0.0347
  effect size r: 0.526

YOLO26m vs YOLOv5m
  p-value: 0.0011
  effect size r: 0.500

